In [5]:
import pandas as pd
from prophet import Prophet
import warnings
import json
from prophet.serialize import model_to_json
import glob

In [2]:
all_files = glob.glob('./data/rainfall/*.csv')
df_list = [pd.read_csv(f) for f in all_files]
df_raw = pd.concat(df_list, ignore_index=True)

station_meta = df_raw[['station_id', 'station_name', 'location_latitude', 'location_longitude']].drop_duplicates('station_id')
station_meta.to_csv('./data/station_meta.csv', index=False)

In [3]:
df_raw['date'] = pd.to_datetime(df_raw['date'])

daily_rain = df_raw.groupby(['station_id', 'date'])['reading_value'].sum().reset_index()
daily_rain['is_rainy'] = (daily_rain['reading_value'] > 0.2).astype(int)

daily_rain['year_month'] = daily_rain['date'].dt.to_period('M').dt.to_timestamp()
monthly_rain = daily_rain.groupby(['station_id', 'year_month'])['is_rainy'].sum().reset_index()

## Rainfall Model

In [6]:
warnings.filterwarnings('ignore')

valid_stations_count = 0
for station_id, group in monthly_rain.groupby('station_id'):
    df_prophet = group[['year_month', 'is_rainy']].rename(columns={'year_month': 'ds', 'is_rainy': 'y'})
    if len(df_prophet) < 12:
        continue

    model_rain_station = Prophet()
    model_rain_station.fit(df_prophet)

    with open(f'./model/model_rain_{station_id}.json', 'w') as fout:
        json.dump(model_to_json(model_rain_station), fout)

    valid_stations_count += 1

print(valid_stations_count)

15:57:20 - cmdstanpy - INFO - Chain [1] start processing
15:57:20 - cmdstanpy - INFO - Chain [1] done processing
15:57:20 - cmdstanpy - INFO - Chain [1] start processing
15:57:20 - cmdstanpy - INFO - Chain [1] done processing
15:57:20 - cmdstanpy - INFO - Chain [1] start processing
15:57:20 - cmdstanpy - INFO - Chain [1] done processing
15:57:20 - cmdstanpy - INFO - Chain [1] start processing
15:57:20 - cmdstanpy - INFO - Chain [1] done processing
15:57:20 - cmdstanpy - INFO - Chain [1] start processing
15:57:20 - cmdstanpy - INFO - Chain [1] done processing
15:57:21 - cmdstanpy - INFO - Chain [1] start processing
15:57:21 - cmdstanpy - INFO - Chain [1] done processing
15:57:21 - cmdstanpy - INFO - Chain [1] start processing
15:57:21 - cmdstanpy - INFO - Chain [1] done processing
15:57:21 - cmdstanpy - INFO - Chain [1] start processing
15:57:21 - cmdstanpy - INFO - Chain [1] done processing
15:57:21 - cmdstanpy - INFO - Chain [1] start processing
15:57:21 - cmdstanpy - INFO - Chain [1]

84


## Sunshine Model

In [7]:
national_monthly_rain = monthly_rain.groupby('year_month')['is_rainy'].mean().reset_index()
national_monthly_rain.rename(columns={'year_month': 'Month', 'is_rainy': 'no_of_rainy_days'}, inplace=True)
national_monthly_rain['Month'] = national_monthly_rain['Month'].dt.strftime('%Y-%m')

In [8]:
df_sun = pd.read_csv('./data/sunshine.csv')
df_sun['month'] = pd.to_datetime(df_sun['month']).dt.strftime('%Y-%m')

df_sun_train = pd.merge(
    national_monthly_rain,
    df_sun,
    left_on='Month',
    right_on='month',
    how='inner'
)

df_prophet_sun = df_sun_train[['Month', 'mean_sunshine_hrs', 'no_of_rainy_days']].rename(columns={
    'Month': 'ds',
    'mean_sunshine_hrs': 'y'
})

In [9]:
model_sun = Prophet()
model_sun.add_regressor('no_of_rainy_days')
model_sun.fit(df_prophet_sun)

with open('./model/model_sun.json', 'w') as fout:
    json.dump(model_to_json(model_sun), fout)

15:59:06 - cmdstanpy - INFO - Chain [1] start processing
15:59:06 - cmdstanpy - INFO - Chain [1] done processing


## Tariff Model

In [10]:
def train_and_save_tariff_model(df, series_name, output_json_name):
    df_filtered = df[df['DataSeries'] == series_name].copy()
    if df_filtered.empty:
        print(f"Warning: No data found for {series_name}")
        return None

    df_long = df_filtered.melt(id_vars=['DataSeries'], var_name='month_str', value_name='price')
    df_long['ds'] = pd.to_datetime(df_long['month_str'], format='%Y%b')
    df_prophet = df_long[['ds', 'price']].rename(columns={'price': 'y'})
    df_prophet = df_prophet.dropna().sort_values('ds').reset_index(drop=True)

    print(f"Training Prophet model for {series_name}...")
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    model.fit(df_prophet)

    future_dates = model.make_future_dataframe(periods=12, freq='MS')
    forecast = model.predict(future_dates)

    with open(output_json_name, 'w') as fout:
        json.dump(model_to_json(model), fout)
    print(f"Model successfully saved as '{output_json_name}'!")

    return df_prophet, forecast

In [11]:
df_all = pd.read_csv('./data/electricity.csv')

# Domestic
dom_data, dom_forecast = train_and_save_tariff_model(
    df=df_all,
    series_name='Low Tension Supplies - Domestic',
    output_json_name='./model/tariff_domestic.json'
)

# Non-Domestic
ind_data, ind_forecast = train_and_save_tariff_model(
    df=df_all,
    series_name='Low Tension Supplies - Non-Domestic',
    output_json_name='./model/tariff_industrial.json'
)

16:00:46 - cmdstanpy - INFO - Chain [1] start processing
16:00:46 - cmdstanpy - INFO - Chain [1] done processing
16:00:46 - cmdstanpy - INFO - Chain [1] start processing
16:00:46 - cmdstanpy - INFO - Chain [1] done processing


Training Prophet model for Low Tension Supplies - Domestic...
Model successfully saved as './model/tariff_domestic.json'!
Training Prophet model for Low Tension Supplies - Non-Domestic...
Model successfully saved as './model/tariff_industrial.json'!
